# Runner — ejecuta todo el pipeline (los 3 casos)

> **Qué hace.** Prepara el entorno reproducible con `uv` (Python 3.13 + dependencias) y ejecuta el pipeline de producción de punta a punta: `kedro run` corre los tres casos (ingesta cruzada → tabla maestra → modelo → salidas) y persiste los modelos en `data/06_models` y las salidas en `data/07_model_output`. Opcionalmente genera los reportes.

> **Cómo usar.** Ábrelo y ejecuta todo de arriba abajo. Un solo archivo, sin comandos manuales.

## Uso en Google Colab

1. Sube el proyecto a Colab (o clónalo si tienes acceso al repositorio) y sitúate en su carpeta.
2. Ejecuta este notebook con **Entorno de ejecución → Ejecutar todo**.

No necesitas instalar Python 3.13 a mano: `uv` lo descarga y aísla. Tampoco necesitas subir los datos por separado —las 12 fuentes ya viven versionadas en `data/01_raw/`—. En local, basta con abrir el notebook (idealmente con `uv run jupyter lab`) y ejecutar todo.

## 1. Preparar el entorno

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

# Raiz del proyecto (funciona desde notebooks/ o desde la raiz).
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
assert (ROOT / 'pyproject.toml').exists(), 'Abre este notebook dentro del proyecto tostao-retail-ml'
print('Proyecto:', ROOT)

# uv gestiona Python 3.13 + dependencias de forma reproducible; se instala si falta (p. ej. en Colab).
if shutil.which('uv') is None:
    print('Instalando uv...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)

def sh(cmd):
    """Ejecuta un comando y transmite su salida al notebook."""
    print('>', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode:
        raise RuntimeError(f'Comando fallo (rc={proc.returncode}): ' + ' '.join(cmd))

In [ ]:
# Entorno reproducible (idempotente): Python 3.13 + dependencias + extra del Caso B.
sh(['uv', 'sync', '--extra', 'caso_b'])

## 2. Ejecutar el pipeline (los 3 casos)

Equivale a `uv run kedro run`. Kedro lee el catálogo, cruza las fuentes en la tabla maestra de cada caso, entrena y evalúa, y persiste modelos y salidas.

In [ ]:
sh(['uv', 'run', 'kedro', 'run'])

## 3. Reportes (opcional)

Genera el reporte **ejecutivo** por caso (`reports/ejecutivo/`) y el **completo** (`data/08_reporting/`). Es un paso pesado; si falla (p. ej. por el motor de imágenes en un entorno restringido) no interrumpe el runner.

In [ ]:
try:
    sh(['uv', 'run', 'python', 'scripts/build_reports.py'])
except Exception as e:
    print('Reportes omitidos (paso opcional):', e)

## 4. Artefactos generados

In [ ]:
def listar(carpeta, patron):
    for p in sorted(Path(carpeta).glob(patron)):
        print(f'  {p}  ({p.stat().st_size/1024:.0f} KB)')

print('Modelos (data/06_models):');       listar('data/06_models', '*.pkl')
print('Salidas (data/07_model_output):'); listar('data/07_model_output', '*')
print('Reportes ejecutivos:');            listar('reports/ejecutivo', '*.html')

## Conclusión

Al terminar, los tres casos están entrenados y evaluados, con sus modelos y salidas persistidos. Para ver el análisis paso a paso, usa `run_notebooks.ipynb` (ejecuta los notebooks de EDA y modelamiento caso por caso) o abre directamente `notebooks/caso_*/`.